# Training a Transformer Language Model

In this notebook, we will build the final component needed to train a standard transformer language model from scratch: the training loop.

**What we'll do:**
1. Implement the cross-entropy loss function and the AdamW optimizer.
2. Implement the training loop, with support for serializing and loading model and optimizer state.

We will build these components from scratch. In particular, we'll avoid using any definitions from `torch.nn`, `torch.nn.functional`, or `torch.optim` except for the following:
- `torch.nn.Parameter`.
- Container classes in `torch.nn` (e.g. `Module`, `ModuleList`, `Sequential`, etc.).
- The `torch.optim.Optimizer` base class.

We now have the steps to tokenize the data and feed it into a transformer model. What remains is to build all of the code to support training. The training portion consists of the following:
- **Loss:** we need to define the loss function (cross-entropy).
- **Optimizer:** we need to define the optimizer to minimize this loss (AdamW).
- **Training loop:** we need all the supporting infrastructure that loads data, saves checkpoints, and manages training.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from einops import rearrange, einsum

import sys; sys.path.insert(0, '..')
from tests.test_training import *

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

device = get_device()
print(f'device = {device}')

## Cross-entropy loss

Recall that the transformer language model defines a distribution $p_\theta(x_{i+1}| x_{1:i})$ for each sequence $x$ of length $m + 1$ and $i = 1, \cdots, m$. Given a training set $D$ consisting of sequences of length $m$, we define the standard cross-entropy (negative log-likelihood) loss function:

$$
\ell(\theta; D) = \frac{1}{|D| m} \sum_{x \in D} \sum_{i=1}^m -\log p_\theta(x_{i+1}| x_{1:i}) \ .
$$

Note that a single forward pass in the transformer yields $p_\theta(x_{i+1}| x_{1:i})$ for *all* $i = 1, \cdots, m$.

In particular, the transformer computes logits $o_i \in \mathbb{R}^{|V|}$ for each position $i$, where $V$ is the vocabulary set, which results in:

$$
p_\theta(x_{i+1}| x_{1:i}) = \text{softmax}(o_i)[x_{i+1}] = \frac{e^{o_i[x_{i+1}]}}{\sum_{j=1}^{|V|} e^{o_i[j]}} \ .
$$

The cross entropy loss is generally defined with respect to the vector of logits $o_i \in \mathbb{R}^{|V|}$ and target $x_{i+1}$. Note that $o_i[k]$ refers to value at index $k$ of the vector $o_i$. This corresponds to the cross entropy between the Dirac delta distribution over $x_{i+1}$ and the predicted $\text{softmax}(o_i)$ distribution.

Implementing the cross entropy loss requires some care with numerical issues, just like in the case of softmax.

### Problem: Implement Cross entropy

Fill in the `cross_entropy` function below. This function should compute the cross entropy loss given a pair of input and target tensors. That is, given features $x_{i+1}$ and targets $o_i$, we want to implement the function

$$
\ell_i = -\log \text{softmax}(o_i)[x_{i+1}] \ .
$$

This function should handle the following:
- Subtract the largest element for numerical stability.
- Cancel out log and exp whenever possible.
- Handle any additional batch dimensions and return the average across the batch. As in previous lessons, we assume batch-like dimensions always come first, before the vocabulary size dimension.

The adapter `run_cross_entropy` will then handle the tests. Be sure the tests are passing before proceeding to the next section.

In [ ]:
def cross_entropy(inputs, targets):
    """
    Given a tensor of inputs and targets, compute the average cross-entropy loss across examples.
    Args:
        inputs (Float[Tensor, 'batch_size vocab_size']): inputs[i][j] is the unnormalized logit of jth class for the ith example.
        targets (Int[Tensor, 'batch_size']): Tensor of shape (batch_size,) with the index of the correct class.
            Each value must be between 0 and num_classes - 1.
    Returns:
        Float[Tensor, '']: The average cross-entropy loss across examples.
    """
    raise NotImplementedError 

In [ ]:
def run_cross_entropy(inputs, targets):
    """
    Adapter for main cross_entropy function
    """
    return cross_entropy(inputs, targets)

test_cross_entropy(run_cross_entropy)

**Perplexity:** Cross entropy suffices for training, but when we evaluate the model, we also want to report perplexity. For a sequence of length $m$ where we suffer cross-entropy losses $\ell_1, \cdots, \ell_m$:

$$
\text{perplexity} = \exp\left( \frac{1}{m} \sum_{i=1}^m \ell_i \right) \ .
$$

## The SGD Optimizer

Now that we have a loss function, we will begin our exploration of optimizers. The simplest gradient-based optimizer is Stochastic Gradient Descent (SGD). We start with randomly initialized parameters $\theta_0$. Then for each step $t = 0, \cdots, T − 1$, we perform the following update:

$$
\theta_{t+1} \leftarrow \theta_t - \alpha_t \nabla L(\theta_t; B_t) \ ,
$$

where $B_t$ is a random batch of data sampled from the dataset $D$, and the learning rate $\alpha_t$ and batch size $|B_t|$ are hyperparameters.

### Implementing SGD in PyTorch

To implement our optimizers, we will subclass the PyTorch `torch.optim.Optimizer` class. An `Optimizer` subclass must implement two methods:
- `def __init__(self, params, ...)` should initialize your optimizer. Here, params will be a collection of parameters to be optimized (or parameter groups, in case the user wants to use different hyperparameters, such as learning rates, for different parts of the model). Make sure to pass params to the `__init__` method of the base class, which will store these parameters for use in step. You can take additional arguments depending on the optimizer (e.g. the learning rate is a common one), and pass them to the base class constructor as a dictionary, where keys are the names (strings) you choose for these parameters.
- `def step(self)`: should make one update of the parameters. During the training loop, this will be called after the backward pass, so you have access to the gradients on the last batch. This method should iterate through each parameter tensor `p` and modify them in place, i.e. setting `p.data`, which holds the tensor associated with that parameter based on the gradient `p.grad` (if it exists), the tensor representing the gradient of the loss with respect to that parameter.

The PyTorch optimizer API has a few subtleties, so it’s easier to explain it with an example. To make our example richer, we’ll implement a slight variation of SGD where the learning rate decays over training, starting with an initial learning rate $\alpha$ and taking successively smaller steps over time:

$$
\theta_{t+1} \leftarrow \theta_t - \frac{\alpha}{\sqrt{t+1}} \nabla L(\theta_t; B_t) \ ,
$$

Let’s see how this version of SGD would be implemented as a PyTorch `Optimizer`:

```python
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f'Invalid learning rate: {lr}')
        defaults = {'lr': lr}
        super().__init__(params, defaults)
    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group['lr'] # Get the learning rate.
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p] # Get state associated with p.
                t = state.get('t', 0) # Get iteration number from the state, or initial value.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state['t'] = t + 1 # Increment iteration number.
        return loss
```

In `__init__`, we pass the parameters to the optimizer, as well as default hyperparameters, to the base class constructor (the parameters might come in groups, each with different hyperparameters). In case the parameters are just a single collection of `torch.nn.Parameter` objects, the base constructor will create a single group and assign it the default hyperparameters. Then, in step, we iterate over each parameter group, then over each parameter in that group, and apply the previous equation. Here, we keep the iteration number as a state
associated with each parameter: we first read this value, use it in the gradient update, and then update it. The API specifies that the user might pass in a callable `closure` to re-compute the loss before the optimizer step. We won’t need this for the optimizers we’ll use, but we add it to comply with the API. 

To see this working, we can use the following minimal example of a *training loop*:

```python
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1)

for t in range(100):
    opt.zero_grad() # Reset the gradients for all learnable parameters.
    loss = (weights**2).mean() # Compute a scalar loss value.
    print(loss.cpu().item())
    loss.backward() # Run backward pass, which computes gradients.
    opt.step() # Run optimizer step.
```

This is the typical structure of a training loop: in each iteration, we will compute the loss and run a step of the optimizer. When training language models, our learnable parameters will come from the model (in PyTorch, `m.parameters()` gives us this collection). The loss will be computed over a sampled batch of data, but the basic structure of the training loop will be the same.

#### Problem: Tuning the learning rate

As we will see, one of the hyperparameters that affects training the most is the learning rate. Let’s see that in practice in our toy example. Run the SGD example above with three other values for the learning rate: `1e1`, `1e2`, and `1e3`, for just 10 training iterations. What happens with the loss for each of these learning rates? Does it decay faster, slower, or does it diverge (i.e. increase over the course of training)?

In [ ]:
# implement code

## AdamW

Modern language models are typically trained with more sophisticated optimizers, instead of SGD. Most optimizers used recently are derivatives of the Adam optimizer (*Kingma and Ba, 2015*). We will use AdamW (*Loshchilov and Hutter, 2019*), which is in wide use in recent work. AdamW proposes a modification to Adam that improves regularization by adding weight decay (at each iteration, we pull the parameters towards zero), in a way that is decoupled from the gradient update. We will implement AdamW as described in algorithm
2 of *Loshchilov and Hutter (2019)*.

AdamW is stateful: for each parameter, it keeps track of a running estimate of its first and second moments. Thus, AdamW uses additional memory in exchange for improved stability and convergence. Besides the learning rate $\alpha$, AdamW has a pair of hyperparameters ($\beta_1$, $\beta_2$) that control the updates to the moment estimates, and a weight decay rate $\lambda$. Typical applications set ($\beta_1$, $\beta_2$) to $(0.9, 0.999)$, but large language models like LLaMA (*Touvron et al., 2023*) and GPT-3 (*Brown et al., 2020*) are often trained with $(0.9, 0.95)$. 

The algorithm can be written as follows, where $\varepsilon$ is a small value (e.g. $10^{−8}$) used to improve numerical stability in case we get extremely small values in $v$:

<center>
<img src="attachment:e366b53c-934a-4e37-9d49-14c3132576f6.png" style="zoom:70%;"/>
</center>

Note that $t$ starts at $1$. You will now implement this optimizer.

### Problem: Implement AdamW

Use the `AdamW` class below to implement the AdamW optimizer. This class subclasses `torch.optim.Optimizer`, and takes in the learning rate $\alpha$, momentum $\beta$, smoothing parameter $\varepsilon$, and weight decay parameter $\lambda$ as constructor arguments.

To help you keep state, the base `Optimizer` class gives you a dictionary `self.state`, which maps `nn.Parameter` objects to a dictionary that stores any information you need for that parameter (for AdamW, this would be the moment estimates). 

Once you've implemented everything, use the adapter `get_adamw_cls` to make sure the tests pass before proceeding.

In [ ]:
class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        """
        Constructor for the AdamW optimizer.
        Args:
            lr: float. Learning rate hyperparameter.
            betas: tuple[float, float]. AdamW momentum hyperparameters.
            eps: float. AdamW epsilon hyperparameter.
            weight_decay: float. Weight decay hyperparameter.
        """
        defaults = {'lr': lr, 'betas': betas, 'eps': eps, 'weight_decay': weight_decay}
        super().__init__(params, defaults)
        raise NotImplementedError
    def step(self, closure=None):
        """
        Perform a single optimization step. That is, perform the AdamW update of θ(t) to θ(t+1).
        Returns:
            loss: float.
        """
        raise NotImplementedError

In [ ]:
def get_adamw_cls():
    """
    Returns a torch.optim.Optimizer that implements AdamW.
    """
    return AdamW

In [ ]:
test_adamw(get_adamw_cls)

### Problem: Resource accounting for training with AdamW

Let us compute how much memory and compute running AdamW requires. Assume we are using `float32` for every tensor.
1. How much peak memory does running AdamW require? Decompose your answer based on the memory usage of the parameters, activations, gradients, and optimizer state. Express your answer in terms of the `batch_size` and the model hyperparameters (`vocab_size`, `context_length`, `num_layers`, `d_model`, `num_heads`). Assume $d_{\mathrm{ff}} = 4 \cdot d_{\mathrm{model}}$. Provide an algebraic expression for each of parameters, activations, gradients, and optimizer state, as well as the total. For simplicity, when calculating memory usage of activations, consider only the following components:
    - transformer block
        - RMSNorm(s)
        - Multi-head self-attention sublayer: QKV projections, Q⊤K matrix multiply, softmax, weighted sum of values, output projection.
        - Position-wise feed-forward: W1 matrix multiply, SiLU, W2 matrix multiply
    - final RMSNorm
    - output embedding
    - cross-entropy on logits
2. Instantiate your answer for a GPT-2 XL-shaped model to get an expression that only depends on the `batch_size`. What is the maximum batch size you can use and still fit within 80GB memory? Provide an expression that looks like `a * batch_size + b` for numerical values `a`, `b`, and a number representing the maximum batch size.
3. How many FLOPs does running one step of AdamW take? Provide an algebraic expression, with a brief justification.
4. Model FLOPs utilization (MFU) is defined as the ratio of observed throughput (tokens per second) relative to the hardware’s theoretical peak FLOP throughput (*Chowdhery et al., 2022*). An NVIDIA A100 GPU has a theoretical peak of 19.5 teraFLOP/s for `float32` operations. Assuming you are able to get 50% MFU, how long would it take to train a GPT-2 XL for 400K steps and a batch size of 1024 on a single A100? Following *Kaplan et al. (2020)* and *Hoffmann et al. (2022)*, assume that the backward pass has twice the FLOPs of the forward pass.
Provide the number of days training would take, with a brief justification.

In [ ]:
# answers

### Learning rate scheduling

The value for the learning rate that leads to the quickest decrease in loss often varies during training. In training transformers, it is typical to use a learning rate schedule, where we start with a bigger learning rate, making quicker updates in the beginning, and slowly decay it to a smaller value as the model trains. For this problem, we will implement the cosine annealing schedule used to train LLaMA (*Touvron et al., 2023*).

A scheduler is simply a function that takes the current step $t$ and other relevant parameters (such as the initial and final learning rates), and returns the learning rate to use for the gradient update at step $t$. The simplest schedule is the constant function, which will return the same learning rate given any $t$.

The cosine annealing learning rate schedule takes (i) the current iteration $t$, (ii) the maximum learning rate $\alpha_{max}$, (iii) the minimum (final) learning rate $\alpha_{min}$, (iv) the number of warm-up iterations $T_w$, and (v) the number of cosine annealing iterations $T_c$. The learning rate at iteration $t$ is defined as:
- **Warm Up:** If $t < T_w$, then
$$\alpha_t = \frac{t}{T_w} \alpha_{max} \ .$$
- **Cosine Annealing:** If $T_w \leq t \leq T_c$, then
$$\alpha_t = \alpha_{min} + \frac{1}{2} (\alpha_{max} - \alpha_{min}) \left[1 + \cos\left(\pi\frac{t - T_w}{T_c - T_w}\right) \right] \ .$$
- **Post Annealing:** If $t > T_c$, then
$$\alpha_t = \alpha_{min} \ .$$

*Note:*  It’s sometimes common to use a schedule where the learning rate rises back up (restarts) to help get past local minima.

#### Problem: Implement cosine learning rate schedule with warmup

Fill in the `lr_cosine_schedule` function below, which takes in the arguments $t$, $\alpha_{max}$, $\alpha_{min}$, $T_w$, and $T_c$, and returns the learning rate $\alpha_t$ according to the scheduler defined above. Then, use the adapter `get_lr_cosine_schedule` to run the tests and make sure they pass before proceeding.

In [ ]:
def lr_cosine_schedule(it, max_learning_rate, min_learning_rate, warmup_iters, cosine_cycle_iters):
    """
    Given the parameters of a cosine learning rate decay schedule (with linear warmup) and an iteration number, return the 
    learning rate at the given iteration under the specified schedule.
    Args:
        it: int. Iteration number to get learning rate for.
        max_learning_rate: float. alpha_max, the maximum learning rate for cosine learning rate schedule (with warmup).
        min_learning_rate: float. alpha_min, the minimum / final learning rate for
            the cosine learning rate schedule (with warmup).
        warmup_iters: int. T_w, the number of iterations to linearly warm-up
            the learning rate.
        cosine_cycle_iters: int. T_c, the number of cosine annealing iterations.
    Returns:
        float. Learning rate at the given iteration under the specified schedule.
    """
    raise NotImplementedError

In [ ]:
def run_get_lr_cosine_schedule(it, max_learning_rate, min_learning_rate, warmup_iters, cosine_cycle_iters):
    """
    Adapter for main lr_cosine_schedule function.
    """
    return lr_cosine_schedule(it, max_learning_rate, min_learning_rate, warmup_iters, cosine_cycle_iters)

test_get_lr_cosine_schedule(run_get_lr_cosine_schedule)

## Gradient clipping

During training, we can sometimes hit training examples that yield large gradients, which can destabilize training. To mitigate this, one technique often employed in practice is *gradient clipping*. The idea is to enforce a limit on the norm of the gradient after each backward pass before taking an optimizer step.

Given the gradient (for all parameters) $g$, we compute its $\ell_2$-norm $||g||_2$. If this norm is less than a maximum value $M$, then we leave $g$ as is; otherwise, we scale $g$ down by a factor of

$$
\frac{M}{||g||_2 + \varepsilon} \ ,
$$

where a small $\varepsilon$, like $10^{−6}$ , is added for numeric stability. Note that the resulting norm will be just under $M$.

### Problem: Implement gradient clipping

Fill in the `gradient_clipping` function given below. This function takes in a list of parameters, and a maximum $\ell_2$-norm to clip the gradient to. For the $\varepsilon$ clipping parameter, we'll use the PyTorch default of `1e-6`.

*Note:* The gradient clipping operation should be done *in-place*. That is, the clipping gets done on `parameters.grad` directly without returning anything.

Once this is finished, use the adapter `run_gradient_clipping` to make sure your implementation passes the tests below.

In [ ]:
def gradient_clipping(parameters, max_l2_norm, eps=1e-6):
    """
    Given a set of parameters, clip their combined gradients to have l2 norm at most max_l2_norm.
    Args:
        parameters: Iterable[torch.nn.Parameter]. Collection of trainable parameters.
        max_l2_norm: float. Positive value containing the maximum l2-norm.
        eps: float = 1e-6. Epsilon for gradient clipping, which defaults to 1e-6.
    The parameter gradients in parameter.grad should be modified in-place.
    """
    raise NotImplementedError

In [ ]:
def run_gradient_clipping(parameters, max_l2_norm):
    """
    Adapter for main gradient_clipping function.
    """
    gradient_clipping(parameters, max_l2_norm)

test_gradient_clipping(run_gradient_clipping)

## Training loop

We will now finally put together the major components we’ve built so far: the tokenized data, the model, and the optimizer.

### Data Loader

The tokenized data, e.g. that you prepared in the tokenization notebook, is a single sequence of tokens $x = (x_1, \cdots, x_n)$. Even though the source data might consist of separate documents (e.g. different web pages, or source code files), a common practice is to concatenate all of those into a single sequence of tokens, adding a delimiter between them (such as the `<|endoftext|>` token).

A data loader turns this into a stream of batches, where each batch consists of $B$ sequences of length $m$, paired with the corresponding next tokens, also with length $m$. For example, for $B = 1$, $m = 3$, $([x_2, x_3, x_4], [x_3, x_4, x_5])$ would be one potential batch.

Loading data in this way simplifies training for a number of reasons. First, any $1 \leq i < n − m$ gives a valid training sequence, so sampling sequences are trivial. Since all training sequences have the same length, there’s no need to pad input sequences, which improves hardware utilization (also by increasing batch size $B$). Finally, we also don’t need to fully load the full dataset to sample training data, making it easy to handle large datasets that might not otherwise fit in memory.

#### Problem: Implement data loading

Fill in the `get_batch` function below. This function takes a numpy array `x` (integer array with token IDs), a `batch_size`, a `context_length`, and a PyTorch device string, and returns a pair of tensors: the sampled input sequences and the corresponding next-token targets. Both tensors should have shape `(batch_size, context_length)` containing token IDs, and both should be placed on the requested device.

*Tip:* When training a model on specific hardware like the GPU or Apple M-series chips, you always need to make sure to move your data to the correct device. It's a good idea to always pass in a `device` argument, which you can default to `'cpu'` if you like. Pass in `'mps'` for Apple M-series chips, and `'cuda'` for GPUs.

*Note:* For more information on MPS, checkout the documentation [here](https://developer.apple.com/metal/pytorch/) and [here](https://pytorch.org/docs/main/notes/mps.html).

Test your implementation using the adapter `run_get_batch`. Try to make sure the test passes before proceeding.

In [ ]:
def get_batch(dataset, batch_size, context_length, device):
    """
    Given a dataset (a 1D numpy array of integers) and a desired batch size and context length, sample language modeling input sequences 
    and their corresponding labels from the dataset.
    Args:
        dataset: np.array. 1D numpy array of integer token IDs in the dataset.
        batch_size: int. Desired batch size to sample.
        context_length: int. Desired context length of each sampled example.
        device: str. PyTorch device string indicating the device to place the sampled input sequences and labels on.
    Returns:
        Tuple of torch.LongTensors of shape (batch_size, context_length). The first tuple item is the sampled input sequences, 
        and the second tuple item is the corresponding language modeling labels.
    """
    raise NotImplementedError 

In [ ]:
def run_get_batch(dataset, batch_size, context_length, device):
    """
    Adapter for the main get_batch function
    """
    return get_batch(dataset, batch_size, context_length, device)

test_get_batch(run_get_batch)

What if the dataset is too big to load into memory? We can use a Unix systemcall named `mmap` which maps a file on disk to virtual memory, and lazily loads the file contents when that memory location is accessed. Thus, you can “pretend” you have the entire dataset in memory. Numpy implements this through `np.memmap` (or the flag `mmap_mode='r'` to `np.load`, if you originally saved the array with `np.save`), which will return a numpy array-like object that loads the entries on-demand as you access them. *When sampling from your dataset (i.e. a numpy array) during training, be sure load the dataset in memory-mapped mode* (via `np.memmap` or the flag `mmap_mode='r'` to `np.load`, depending on how you saved the array). Make sure you also specify a `dtype` that matches the array that you’re loading. It may be helpful to explicitly verify that the memory-mapped data looks correct (e.g. doesn’t contain values beyond the
expected vocabulary size).

### Checkpointing

In addition to loading data, we will also need to save models as we train. When running jobs, we often want to be able to resume a training run that for some reason stopped midway (e.g. due to your job timing out, machine failure, etc). Even when all goes well, we might also want to later have access to intermediate models (e.g. to study training dynamics post-hoc, take samples from models at different stages of training, etc).

A checkpoint should have all the states that we need to resume training. We of course want to be able to restore model weights at a minimum. If using a stateful optimizer (such as AdamW), we will also need to save the optimizer’s state (e.g. in the case of AdamW, the moment estimates). Finally, to resume the learning rate schedule, we will need to know the iteration number we stopped at. PyTorch makes it easy to save all of these: every `nn.Module` has a `state_dict()` method that returns a dictionary with all learnable weights; we can restore these weights later with the sister method `load_state_dict()`. The same goes for any `nn.optim.Optimizer`. Finally, `torch.save(obj, dest)` can dump an object (e.g. a dictionary containing tensors in some values, but also regular Python objects like integers) to a file (path) or file-like object, which can then be loaded back into memory with `torch.load(src)`.

#### Problem: Implement model checkpointing

Implement the two functions in the cells below to load and save checkpoints:
- `save_checkpoint` should dump all the state from the first three parameters into the file-like object out. You can use the `state_dict` method of both the model and the optimizer to get their relevant states and use `torch.save(obj, out)` to dump `obj` into out (PyTorch supports either a path or a file-like object here). A typical choice is to have `obj` be a dictionary, but you can use whatever format you want as long as you can load your checkpoint later.
- `load_checkpoint` should load a checkpoint from `src` (path or file-like object), and then recover the model and optimizer states from that checkpoint. Your function should return the iteration number that was saved to the checkpoint. You can use
`torch.load(src)` to recover what you saved in your `save_checkpoint` implementation, and the `load_state_dict` method in both the model and optimizers to return them to their previous states.

Use the adapters `run_save_checkpoint` and `run_load_checkpoint` to run the tests. Make sure they pass before moving on.

In [ ]:
def save_checkpoint(model, optimizer, iteration, out):
    """
    Given a model, optimizer, and an iteration number, serialize them to disk.
    Args:
        model: torch.nn.Module. Serialize the state of this model.
        optimizer: torch.optim.Optimizer. Serialize the state of this optimizer.
        iteration: int. Serialize this value, which represents the number of training iterations
            we've completed.
        out: (str | os.PathLike | BinaryIO | IO[bytes]). Path or file-like object to serialize the model, optimizer, and iteration to.
    """
    raise NotImplementedError

In [ ]:
def load_checkpoint(src, model, optimizer):
    """
    Given a serialized checkpoint (path or file-like object), restore the serialized state to the given model and optimizer. 
    Return the number of iterations that we previously serialized in the checkpoint.
    Args:
        src: (str | os.PathLike | BinaryIO | IO[bytes]). Path or file-like object to serialized checkpoint.
        model: torch.nn.Module. Restore the state of this model.
        optimizer: torch.optim.Optimizer. Restore the state of this optimizer.
    Returns:
        int: the previously-serialized number of iterations.
    """
    raise NotImplementedError

In [ ]:
def run_save_checkpoint(model, optimizer, iteration, out):
    """
    Adapter for the main save_checkpoint function
    """
    return save_checkpoint(model, optimizer, iteration, out)


def run_load_checkpoint(src, model, optimizer):
    """
    Adapter for the main load_checkpoint function
    """
    return load_checkpoint(src, model, optimizer)

test_checkpointing(get_adamw_cls, run_save_checkpoint, run_load_checkpoint)

### Training loop

Now, it’s finally time to put all of the components you implemented together into your main training loop. It will pay off to make it easy to start training runs with different hyperparameters, e.g. by taking them as command-line arguments, since we'll be doing this kind of thing many times later to study how different choices impact training.

#### Problem: Put it together

Write a training loop to train your model on user-provided input. In particular, we recommend that your training loop use most or all of the following:
- Ability to configure and control the various model and optimizer hyperparameters.
- Memory-efficient loading of training and validation large datasets with `np.memmap`.
- Serializing checkpoints to a user-provided path.
- Periodically logging training and validation performance (e.g. to console and/or an external service like [Weights and Biases](https://wandb.ai/)).

In [ ]:
# implement code

## Generating text

Now that we can train models, the last piece we need is the ability to generate text from our model. Recall that a language model takes in a (possibly batched) integer sequence of length (`sequence_length`) and produces a matrix of size `(sequence_length, vocab_size)`, where each element of the sequence is a probability distribution predicting the next word after that position. We will now write a few functions to turn this into a sampling scheme for new sequences.

**Softmax:** By standard convention, the language model output is the output of the final linear layer (the "logits") and so we have to turn this into a normalized probability via the softmax operation, which we saw earlier.

**Decoding:** To generate text (decode) from our model, we will provide the model with a sequence of prefix tokens (the "prompt"), and ask it to produce a probability distribution over the vocabulary that predicts the next word in the sequence. Then, we will sample from this distribution over the vocabulary items to determine the next output token.

Concretely, one step of the decoding process should take in a sequence $x_{1 \cdots t}$ and return a token $x_{t+1}$ via the following equation,

$$
P(x_{t+1} = i | x_{1 \cdots t}) = \frac{e^{v_i}}{\sum_{j=1}^{|V|} e^{v_j}} \ ,
$$

where each logit $v \in \mathbb{R}^{|V|}$ is obtained by doing a forward pass through the LM,

$$
v = \text{TransformerLM}(x_{1 \cdots t})_t \ .
$$

That is, $\text{TransformerLM}$ is our model which takes as input a sequence of `sequence_length` and produces a matrix of size `(sequence_length, vocab_size)`, and we take the last element of this matrix, as we are looking for the next word prediction at the $t$-th position.

This gives us a basic decoder by repeatedly sampling from these one-step conditionals (appending our previously-generated output token to the input of the next decoding timestep) until we generate the end-ofsequence token `<|endoftext|>` (or a user-specified maximum number of tokens to generate).

**Decoder tricks:** We will be experimenting with small models, and small models can sometimes generate very low quality texts. Two simple decoder tricks can help fix these issues. First, in temperature scaling we modify our softmax with a temperature parameter $\tau$, where the new softmax is

$$
\text{softmax}(v; \tau)_i = \frac{e^{v_i/\tau}}{\sum_{j=1}^{|V|} e^{v_j/\tau}} \ .
$$

Note how setting $\tau \to 0$ makes it so that the largest element of $v$ dominates, and the output of the softmax becomes a one-hot vector concentrated at this maximal element.

Second, another trick is *nucleus* or *top-p* sampling, where we modify the sampling distribution by truncating low-probability words. Let $q$ be a probability distribution that we get from a (temperature-scaled) softmax of size (`vocab_size`). Nucleus sampling with hyperparameter $p$ produces the next token according to the equation

$$
P(x_{t+1} = i | q) = \begin{cases}
q_i \big/ \sum_{j \in V(p)} q_j & i \in V(p) \\
0 & \text{otherwise}
\end{cases} \ ,
$$

where $V(p)$ is the *smallest set* of indices such that $\sum_{j \in V(p)} q_j \geq p$. You can compute this quantity easily by first sorting the probability distribution $q$ by magnitude, and selecting the largest vocabulary elements until you reach the target level of $\alpha$.

### Problem: Decoding

Implement a function to decode from your language model. We recommend having support for most or all of the following features:
- Generate completions for a user-provided prompt, i.e. take in some $x_{1 \cdots t}$ and sample a completion until you hit an `<|endoftext|>` token.
- Allow the user to control the maximum number of generated tokens.
- Given a desired temperature value, apply softmax temperature scaling to the predicted next-word distributions before sampling.
- Top-p sampling (*Holtzman et al., 2020*; also referred to as nucleus sampling), given a user-specified threshold value.

In [ ]:
# implement code